In [1]:
# 01_prediction_signal.ipynb
# Module 1: out-of-sample predictive-signal asymmetry between provisioning (resource)
# and participation (activity) blocks for Computer & Information Literacy (CIL).
import warnings; warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np, pandas as pd, pyreadstat
import seaborn as sns, matplotlib.pyplot as plt
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import GroupKFold
# --- grayscale publication style ---
sns.set_theme(style="whitegrid", context="paper")
plt.rcParams.update({"axes.edgecolor":"0.15","grid.color":"0.85","font.size":10,
                     "axes.titlesize":1,"figure.autolayout":True})
GRAY = ["0.20","0.45","0.65","0.80"]
# --- resolve folders relative to repo (no path is printed, for anonymity) ---
BASE = Path("..") if (Path("../data")).exists() else Path(".")
DATA, FIG, TAB = BASE/"data", BASE/"results/figures", BASE/"results/tables"
FIG.mkdir(parents=True, exist_ok=True); TAB.mkdir(parents=True, exist_ok=True)
def savefig(fig, name):
    # save both formats at 600 dpi, no caption/title; paths are never printed
    for ext in ("png","pdf"): fig.savefig(FIG/(name+"."+ext), dpi=600, bbox_inches="tight")
    plt.close(fig)
PVS=[f"PV{i}CIL" for i in range(1,6)]
def wR2(y,yh,w):
    yb=np.average(y,weights=w); return 1-np.sum(w*(y-yh)**2)/np.sum(w*(y-yb)**2)

# load primary case (one country) student + school files and merge school ICT resources
stu,_=pyreadstat.read_sas7bdat(str(DATA/"bsgkori3.sas7bdat"))
sch,_=pyreadstat.read_sas7bdat(str(DATA/"bcgkori3.sas7bdat"))
RES_S=["C_ICTDEV","C_ICTSTD","C_RATDEV","C_RATSTD","C_RATSMB","C_ICTRES","C_RESTOOL","C_RESUTIL","C_HINRES"]
df=stu.merge(sch[["IDCNTRY","IDSCHOOL"]+[c for c in RES_S if c in sch]],on=["IDCNTRY","IDSCHOOL"],how="left")
# two predictor blocks (socio-demographic subgroup vars are deliberately excluded here)
RESOURCE=[c for c in RES_S if c in df]+["S_INTNET"]                 # provisioning / access
PARTICIP=["S_EXCOMP","S_ICTSTUD","S_LRNINTS","S_LRNINTO","S_ACMULT","S_GENEFF","S_SPECEFF",
          "S_CTCLS","S_LRNSAFE","S_GENCLASS","S_SPECLASS"]          # engagement / activity
df=df[df.TOTWGTS.notna() & df.IDSCHOOL.notna()].copy()
def oos(cols):
    # pooled over 5 plausible values x school-clustered folds, weighted out-of-sample R^2
    X=df[cols].to_numpy(float); w=df.TOTWGTS.to_numpy(); g=df.IDSCHOOL.to_numpy(); gkf=GroupKFold(5); s=[]
    for pv in PVS:
        y=df[pv].to_numpy(float); f=[]
        for tr,te in gkf.split(X,y,groups=g):
            m=HistGradientBoostingRegressor(max_iter=300,learning_rate=0.05,max_leaf_nodes=31,
                l2_regularization=1.0,random_state=0).fit(X[tr],y[tr],sample_weight=w[tr])
            f.append(wR2(y[te],m.predict(X[te]),w[te]))
        s.append(np.mean(f))
    return np.mean(s)
res=pd.DataFrame({"block":["Resource/access","Participation/activity","Both combined"],
                  "oos_wR2":[oos(RESOURCE),oos(PARTICIP),oos(RESOURCE+PARTICIP)]})
res.to_csv(TAB/"m1_oos_r2.csv",index=False)
print(res.round(4).to_string(index=False))
# figure: grayscale bar of out-of-sample R^2 by block (no title, no caption)
fig,ax=plt.subplots(figsize=(4.2,3.2))
ax.bar(res.block, res.oos_wR2, color=GRAY[:3], edgecolor="black", linewidth=0.6)
ax.set_ylabel("Out-of-sample weighted $R^2$"); ax.set_xlabel("")
ax.set_xticklabels(["Resource","Participation","Both"], rotation=0)
savefig(fig,"fig_m1_signal")


                 block  oos_wR2
       Resource/access   0.0137
Participation/activity   0.2471
         Both combined   0.2389
